<a href="https://colab.research.google.com/github/frbinucci/DNN_Splitting/blob/main/DNNSplitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DNN Splitting

This notebook allows to evaluate the noise performance degradation on the DNN splitting. We consider the [**Intel Image Classification**](https://www.kaggle.com/datasets/puneet6060/intel-image-classification) dataset, composed of **25k** landscape images to be classified in **6** different categories.

The model considered for the DNN splitting is the well-known **MobileNetV2**, one of the most popular lightweight models for computer vision tasks.

# Libraries Import and Data Loading

The following code cells allow to import all the necessary libraries and to load the data on the notebook local filesystem from Google Drive.

In [ ]:
#Filesystem management
from google.colab import drive
import zipfile
import os

#Image management
from PIL import Image, ImageFile

#Deep Learning Modules
import torch
from torch.utils.data import dataset
from torchvision.transforms import transforms
from torch import optim
from torch import nn

#Utils
import argparse
import copy

#Numpy
import numpy as np

#Plotting
import matplotlib.pyplot as plt

#Global variables

temporary_output = None
import scipy

# Filesystem Mounting

In order to execute the script:

Go to: https://drive.google.com/drive/folders/1ypC-QtPDBifasdqO6ejXx55Qm7I9zGND?usp=drive_link and copy the file "archive.zip" in a path in your Google Drive.
Change the string "/content/gdrive/MyDrive/PhDQuantumProject/archive.zip" in the following code to "/content/gdrive/MyDrive/path_to_zip_file.

In [ ]:
drive.mount('/content/gdrive',force_remount=True)

drive.mount('/content/gdrive')
zip_ref = zipfile.ZipFile("/content/gdrive/MyDrive/DNNSplitting/archive.zip", 'r')
zip_ref.extractall("/content/dataset")
zip_ref.close()


# Dataset Class
This class is used to load the Intel Image Classification Dataset from the local storage once it has been loaded on the Notebook local filesystem.

In [ ]:
class LandscapeDataset(dataset.Dataset):
  def __init__(self,**kwargs):
    super(LandscapeDataset,self).__init__()

    self.data_root = kwargs.get('data_root','./dataset/archive')
    self.data = list()
    self.train_or_test = kwargs.get('train_or_test','train')

    self.str2num = {'buildings':0,'forest':1,'glacier':2,'mountain':3,'sea':4,'street':5}

    training_path = os.path.join(self.data_root,'seg_train')
    test_path = os.path.join(self.data_root,'seg_test')

    self.transforms = transforms.Compose([
      transforms.Resize(256),
      transforms.CenterCrop(224),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    if self.train_or_test == 'train':
      dataset_path = training_path
    else:
      dataset_path = test_path

    for current_directory in os.listdir(dataset_path):
        for current_photo in os.listdir(os.path.join(dataset_path,current_directory)):
            image = Image.open(os.path.join(dataset_path,current_directory,current_photo))
            image_data_tensor = self.transforms(image)
            label = self.str2num[current_directory]
            self.data.append([image_data_tensor,label])
      #This method must be overrided. It returns information about features and labels.
  def __getitem__(self, item):
    return self.data[item][0],int(self.data[item][1])

  #This method must be override. It returns information about length of data vector
  def __len__(self):
      return len(self.data)



# Class separability Metrics

# Solver Class

The following class is used to train the network and to evaluate the performance on the test-set, taking into account the DNN splitting philosophy.The implemented features are **(WIP)**:


1.   Splitting point selection
2.   Noise injection
3.   Testing with noise for a given splitting point and an (average) noise power
4.   Training with noise.

In [ ]:
class Solver():
    def __init__(self, **kwargs):

        self.saving_point = kwargs.get('saving_point',None)
        self.model = kwargs.get('model',None)
        self.epoch_number = kwargs.get('max_epochs', 50)
        self.model_output_dir = kwargs.get('ckpt_dir', './model')
        self.image_size = kwargs.get('image_size', 256)
        self.batch_size = kwargs.get('batch_size',1)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.data_root = kwargs.get('data_root', './dataset/archive')
        self.power_noise = kwargs.get('power_noise',None)

        self.optimizer = optim.Adam(list(self.model.parameters()), lr=1e-3)

        self.criterion = nn.CrossEntropyLoss()
        self.ber = kwargs.get('ber', 0)
        self.noise_power = kwargs.get('noise_power', 0)

        self.transform = transforms.ToTensor()

    def load_dataset(self,**kwargs):
        load_train = kwargs.get('load_train',True)
        load_test = kwargs.get('load_test',True)

        if load_train == True:
          self.training_loader = torch.utils.data.DataLoader(LandscapeDataset(data_root=self.data_root,train_or_test='train'),
                                                           batch_size=self.batch_size,
                                                           shuffle=True,
                                                           num_workers=2,
                                                           pin_memory=True)
        if load_test == True:
          self.test_loader = torch.utils.data.DataLoader(LandscapeDataset(data_root=self.data_root,train_or_test='test'),
                                                       batch_size=self.batch_size,
                                                       shuffle=True,
                                                       num_workers=2,
                                                       pin_memory=True)

    def fit(self, **kwargs):
        self.load_dataset()
        self.model.to(self.device)
        if not os.path.exists(self.model_output_dir):
            os.makedirs(self.model_output_dir)

        early_stopping = kwargs.get('early_stopping', False)
        best_test_loss = float('inf')
        best_test_accuracy = 0
        # torch.manual_seed(42)
        for epoch_index in range(self.epoch_number):
            running_loss = 0
            self.model.train()
            for batch, (input, labels) in enumerate(self.training_loader):
                # Zero gradient
                self.optimizer.zero_grad()

                input = input.to(self.device)
                labels = labels.to(self.device)
                output = self.model(input)

                # Computing loss
                loss = self.criterion(output, labels)
                # Backpropagation
                loss.backward()
                # Step forward
                self.optimizer.step()

                size = len(self.training_loader.dataset)
                # Computing mean loss
                running_loss += loss.item()
                if (batch+1) % 100 == 0:
                    loss, current = loss.item(), batch * len(input)
                    print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

            #self.autoencoder.setDecoder(False)
            test_accuracy,_ = self.evaluate(self.test_loader)
            train_accuracy,_ = self.evaluate(self.training_loader)

            print("Epoch [{}/{}], Train acc Acc: {:.3f}".format(epoch_index + 1, self.epoch_number, train_accuracy))

            if self.saving_point==None:
              if test_accuracy>best_test_accuracy:
                  print("Epoch [{}/{}], Validation Acc: {:.3f}".format(epoch_index + 1, self.epoch_number, test_accuracy))
                  best_test_accuracy = test_accuracy
                  self.save_model()
            else:
              if test_accuracy>self.saving_point:
                print("Epoch [{}/{}], Validation Acc: {:.3f}".format(epoch_index + 1, self.epoch_number, test_accuracy))
                best_test_accuracy = test_accuracy
                self.save_model()
            if (epoch_index+1)%10==0:
              self.save_model(epoch_index=epoch_index+1)
        return best_test_accuracy

    def compute_separation_metrics(self,size,n_classes,loader):
      list_array = np.empty((n_classes,),dtype=object)
      list_array = [[] for _ in range(list_array.shape[0])]
      average_vector = np.zeros(size)
      self.model.eval()
      item_counter = 0
      with torch.no_grad():
        for item in loader:
          input,labels=item
          labels = labels.to(self.device)
          input = input.to(self.device)

          output = self.model(input)
          out_np = torch.squeeze(temporary_output).cpu().detach().numpy().flatten()
          list_array[torch.squeeze(labels)].append(out_np)

          average_vector+=out_np
          item_counter+=1
      average_vector/=item_counter

      print(f'Item counter={item_counter}')
      ssb = np.zeros((size,size))
      ssw_final = np.zeros((size,size))
      for class_vector in list_array:
        mean_class = np.mean(class_vector,axis=0)
        avg_diff = (average_vector-mean_class).reshape(-1, 1)
        nk = len(class_vector)
        ssb += nk*np.matmul(avg_diff,avg_diff.T)

        ssw = np.zeros((size,size))
        for vec in class_vector:
          vector = (vec-mean_class).reshape(-1, 1)
          ssw+=np.matmul(vector,vector.T)
        ssw_final+=ssw

      ssb/=item_counter
      ssw_final/=item_counter

      fuzziness = np.trace(np.matmul(ssw_final,scipy.linalg.pinv(ssb)))

      return fuzziness

    def save_model(self,**kwargs):
        save_name='landscape_classifier'
        epoch_index=kwargs.get('epoch_index',None)
        if not epoch_index==None:
          save_name = f'network_{epoch_index}.pth'
        print(f"Saving model at: {os.path.join(self.model_output_dir, save_name)}")
        if self.model != None:
            torch.save(self.model.state_dict(), os.path.join(self.model_output_dir, save_name))
    # Method used in order to implement early stopping
    def evaluate(self, loader):
        activations = {}
        avg_flippling_time = 0
        num_correct = 0
        avg_entropy = 0
        num_total = 0
        self.model.eval()
        running_loss = 0
        mean_encoding_time = 0
        valid_list = None
        m = torch.nn.Softmax()
        with torch.no_grad():
            for item in loader:
                input, labels = item

                labels = labels.to(self.device)
                input = input.to(self.device)

                if self.power_noise!=None:
                  power = torch.sum(torch.pow(input, 2))/torch.numel(input)
                  input/=torch.sqrt(power)
                  input = input + np.sqrt(self.power_noise)*torch.randn(input.shape).to(self.device)
                  input*=torch.sqrt(power)

                output = self.model(input)

                entropy = -torch.sum(m(output.detach())*torch.log2(m(output.detach())))
                if (not torch.isnan(entropy)) and (torch.isfinite(entropy)):
                  avg_entropy+=entropy
                _, preds = torch.max(output.detach(), 1)
                num_correct += (preds == labels).sum().item()
                num_total += labels.size(0)
                loss = self.criterion(output, labels)
                running_loss += loss.item()
                running_loss /= len(loader)
        correct = (num_correct / num_total)
        avg_entropy =(avg_entropy/num_total)
        #print(f"Test Error: \n Accuracy: {(100 * correct):>0.1f}%, Avg loss: {running_loss:>8f} \n")

        return correct,avg_entropy

    def get_test_loader(self):
        return self.test_loader

    def set_model(self, model):
        self.model = model

    def test(self,**kwargs):
      index = kwargs.get('index',None)
      self.load_dataset(load_train=False)
      accuracy,entropy = self.evaluate(self.test_loader)
      return accuracy,entropy

    def compute_fuzziness(self,**kwargs):
      n_classes = kwargs.get('index',6)
      size = kwargs.get('size',25088)

      self.load_dataset(load_train=False)
      fuzziness = self.compute_separation_metrics(size,n_classes,self.test_loader)
      return fuzziness

# Main

The following code allows to start the training/test procedure. The parameters are specified with the argparser libary.

In [ ]:
#Global variables
average_power = 0
input_counter = 0

def getIntermediateOutput(name,device):
  # the hook signature
  def hook(model, input, output):
    global temporary_output
    temporary_output = output
    return output
  return hook


def corruptActivation(name,power_noise,device):
  # the hook signature
  def hook(model, input, output):
    #output = output.detach()
    power = torch.sum(torch.pow(output, 2))/torch.numel(output)
    output/=torch.sqrt(power)
    output = output + np.sqrt(power_noise)*torch.randn(output.shape).to(device)
    output*=torch.sqrt(power)
    return output
  return hook

def estimateAveragePower(name):
  # the hook signature
  def hook(model, input, output):
    output = output.detach()
    global average_power
    global input_counter
    average_power = average_power+torch.sum(torch.pow(output, 2))/torch.numel(output)
    input_counter = input_counter + 1
    return output
  return hook

def train():

    drive.mount('/content/gdrive',force_remount=True)
    parser = argparse.ArgumentParser()

    parser.add_argument("--device",type=str,default="cuda")

    parser.add_argument("--model_to_test",type=str,default='mobilenet_v2')
    parser.add_argument("--mode",type=str,default='test',help="Operational Mode. 'train' is used to train the network, 'test' to evalute the noise effects on the early stopping")
    parser.add_argument("--snr",type=int,default=-5,help="The Average SNR to be considered in the evaluation process")

    parser.add_argument("--batch_size", type=int, default=32,help="Batch size for training")
    parser.add_argument("--max_epochs", type=int, default=300,help="Maximum number of epochs ")

    # Training tracking arguments
    parser.add_argument("--print_every", type=int, default=1)
    parser.add_argument("--print_every_minibatches", type=int, default=300)

    parser.add_argument("--ckpt_dir",type=str,default="/content/gdrive/MyDrive/DNNSplitting/BestNetworks/OverTrainingTest/LR3DP05Corrected",help="Path on which save the best classifier")
    parser.add_argument("--path_under_test",type=str,default="/content/gdrive/MyDrive/DNNSplitting/BestNetworks/OverTrainingTest/LR3DP05/network_30.pth",help="Path of the model on which evaluate the effect of the noise")
    parser.add_argument("--results_output_dir",type=str,default="/content/gdrive/MyDrive/DNNSplitting/SandBox",help="Ouptut dir on which save the simulation results")
    #parser.add_argument("--entropy",type=str,default="/content/gdrive/MyDrive/DNNSplitting/Results/corrected_entropy_network_30_5New.npy",help="Ouptut dir on which save the simulation results")

    args, unknown = parser.parse_known_args()

    sim_path = args.results_output_dir
    mode = args.mode
    pure_data_testing = args.pure_data_testing
    model_to_test = args.model_to_test

    classifier = torch.hub.load('pytorch/vision:v0.10.0', model_to_test, pretrained=True)

    features_layers_list = list(classifier.features.children())
    classifier.classifier[0] = nn.Dropout(p=0.5)
    classifier.classifier[1] = nn.Linear(1280, 6)

    classifier_list = list(classifier.classifier.children())

    print("***Network Architecture***")
    for layer in classifier_list:
      print(layer)
    print("**************************")


    if mode == 'train':
        solver = Solver(model=classifier,ckpt_dir=args.ckpt_dir,batch_size=args.batch_size,max_epochs=args.max_epochs)
        solver.fit()
    elif mode == 'test':
        lut_array = np.linspace(0,18,19).astype(np.int)
        lut_array = [0]
        noise_power_db = -args.snr
        print(f"##### Simulations for SNR = {args.snr} dB #####")

        noise_realizations = 1
        accuracy_data = np.zeros(len(lut_array))
        entropy_data = np.zeros(len(lut_array))
        index = 0
        for lut in lut_array:

            mean_accuracy = 0
            mean_entropy = 0
            print(f'Simulating splitting point n. {lut}...')
            for i in range(0,noise_realizations):
              classifier = torch.hub.load('pytorch/vision:v0.10.0', model_to_test, pretrained=False,verbose=False)
              classifier.classifier[1] = nn.Linear(1280, 6)
              path_to_test = args.path_under_test
              classifier.load_state_dict(torch.load(path_to_test))
              classifier = classifier.to(args.device)


              noise_power = 10**(noise_power_db/10)

              if lut>0:
                classifier.features[lut-1].register_forward_hook(corruptActivation(f'l{lut-1}',noise_power,'cuda'))
                raw_noise_power = None
              else:
                raw_noise_power = noise_power

              solver = Solver(model=classifier,batch_size=args.batch_size,power_noise=raw_noise_power)
              accuracy,entropy = solver.test(index=(lut+1))

              print(f'Realization n. [{i+1}/{noise_realizations}]')
              mean_accuracy+=accuracy
              mean_entropy+=entropy
            mean_accuracy/=noise_realizations
            mean_entropy/=noise_realizations
            accuracy_data[lut] = mean_accuracy
            entropy_data[lut] = mean_entropy
            index = index+1
        print(f'Mean accuracy = {mean_accuracy}')
        np.save(f'{sim_path}/average_accuracy_snr_{args.snr}.npy',accuracy_data)
        np.save(f'{sim_path}/average_entropy_snr_{args.snr}.npy',entropy_data)

if __name__ == '__main__':
    train()
